[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_02_data_structures_algorithms/ch11_algorithms_complexity/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 11 Hands-On Coding Exam — Part II Capstone Algorithmic Suite (100 Points)

**Book Reference:** Chapter 11, Section 11.7 (Part II Capstone Coding Exam)  
**Suggested Time:** 40–50 minutes (Open Book, Closed Solution)

| Task | Algorithmic Public Health Function | Points |
| :--- | :--- | :---: |
| **Task 11.1** | `match_lab_results_fast(patients, lab_results)` ($O(n + m)$ hash-join, non-mutating) | 30 |
| **Task 11.2** | `rank_districts_by_burden(district_records)` (Guard filter + multi-key sort) | 35 |
| **Task 11.3** | `trace_exposure_depth(contact_tree, person_id)` (Recursive max transmission depth) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import sharukh_tutor as tutor

tutor.activate(chapter="ch11", notebook="coding_exam")


---
## Exam Task 11.1 (30 Points): Fast $O(n + m)$ Hash-Join Lab Matcher

Write `match_lab_results_fast(patients, lab_results)`:
1. `patients` is a list of patient dicts, each containing `"patient_id"` (plus other fields like `"district"`).
2. `lab_results` is a list of lab dicts, each containing `"patient_id"` and `"hb_g_dl"`.
3. Build a dictionary index from `lab_results` in $O(m)$ time, then return a **new list of new dicts** where each patient dict includes `"hb_g_dl"` from `lab_results` (or `None` if no lab record exists for that `patient_id`).
4. Do **not** mutate any input list or dict. Your function is benchmarked on **3,000 rows** so $O(n^2)$ nested loops will be flagged!

In [ ]:
def match_lab_results_fast(patients, lab_results):
    """Hash-join patients with lab hb_g_dl in O(n + m) without mutating."""
    lab_map = {row["patient_id"]: row["hb_g_dl"] for row in lab_results}
    merged = []
    for patient in patients:
        record = dict(patient)
        record["hb_g_dl"] = lab_map.get(patient["patient_id"], None)
        merged.append(record)
    return merged


---
## Exam Task 11.2 (35 Points): District Disease-Burden Ranker

Write `rank_districts_by_burden(district_records)` where each record is a dict with keys `"district"`, `"cases"`, and `"population"`:
1. Keep only districts with `population > 0` (skipping zero or negative populations cleanly).
2. Compute `incidence_per_100k = round((cases / population) * 100000, 2)`.
3. Return a list of `(district, incidence_per_100k)` tuples sorted by:
   - `incidence_per_100k` **descending** (highest burden first)
   - `district` name **ascending** (alphabetical tie-breaker when rates are equal).

In [ ]:
def rank_districts_by_burden(district_records):
    """Return (district, incidence_per_100k) sorted desc rate, asc name."""
    ranked = []
    for row in district_records:
        if row["population"] <= 0:
            continue
        rate = round((row["cases"] / row["population"]) * 100000, 2)
        ranked.append((row["district"], rate))
    return sorted(ranked, key=lambda item: (-item[1], item[0]))


---
## Exam Task 11.3 (35 Points): Recursive Exposure-Chain Depth Tracer

Write `trace_exposure_depth(contact_tree, person_id)` that recursively computes the **maximum transmission chain depth** (number of transmission hops) starting from `person_id`:
- **Base case:** If `person_id` has no onward contacts (`contact_tree.get(person_id, [])` is empty), return `0`.
- **Recursive step:** Return `1 + max(trace_exposure_depth(contact_tree, c) for c in contacts)`.

In [ ]:
def trace_exposure_depth(contact_tree, person_id):
    """Recursively return max transmission hops starting from person_id."""
    contacts = contact_tree.get(person_id, [])
    if not contacts:
        return 0
    return 1 + max(
        trace_exposure_depth(contact_tree, c) for c in contacts
    )


In [ ]:
exam_submissions = {
    "ch11_exam_01": match_lab_results_fast,
    "ch11_exam_02": rank_districts_by_burden,
    "ch11_exam_03": trace_exposure_depth,
}

tutor.check_exam("ch11", exam_submissions)


---
## Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 11.1 (Hash-Join & Immutability):** Nested loops (`for p in patients: for r in lab_results:`) do 9 million iterations on 3,000 rows ($O(n \times m)$). Building `lab_map` once in $O(m)$ makes every lookup $O(1)$, and `dict(patient)` prevents mutating the caller's dicts.
- **Task 11.2 (`reverse=True` Tie-Breaker Trap):** Sorting with `key=lambda x: (x[1], x[0]), reverse=True` accidentally reverses the district tie-breaker (`"Khulna"` before `"Barishal"`). Using `key=lambda item: (-item[1], item[0])` keeps incidence descending and district names ascending.
- **Task 11.3 (`max()` on Empty List & Hop vs. Node Count):** Calling `max(...)` before checking `if not contacts: return 0` raises `ValueError: max() iterable argument is empty`. Returning `0` for leaves gives exact transmission hops (`1` for a direct contact, `0` for an isolated case).